# Bus passenger count

Notebook goals:
- run full train + eval pipeline
- optionally skip training and evaluate base/raw weights
- keep all frequent experiment knobs in one config cell

## 1) Setup

Strict setup: this notebook must run from the project's `notebooks/` directory.

It loads `.env`, checks CUDA/imports, and logs into wandb.

If you changed `.env`, restart kernel and run from the top.

In [7]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    raise RuntimeError(
        "Run this notebook from the project's 'notebooks/' directory."
    )

PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC = (PROJECT_ROOT / "src").resolve()

# Keep src first in sys.path exactly once.
sys.path = [str(SRC), *[p for p in sys.path if p != str(SRC)]]

# Refresh env vars before any wandb init.
load_dotenv(PROJECT_ROOT / ".env", override=True)

import torch
from ultralytics import YOLO
import wandb

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only)")
print("WANDB_ENTITY:", os.environ.get("WANDB_ENTITY"))
print("WANDB_PROJECT:", os.environ.get("WANDB_PROJECT"))
print("YOLO import: ok", YOLO)

wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


CUDA: True NVIDIA GeForce RTX 3090
WANDB_ENTITY: cristianmaza
WANDB_PROJECT: bus-passenger-count
YOLO import: ok <class 'ultralytics.models.yolo.model.YOLO'>


False

## 2) Experiment Config

Set model/training params and data/preprocessing selection here.

In [ ]:
import config

# Model options: "yolo" (fine-tune) or "yolo_raw" (base model eval)
MODEL_NAME = "yolo"
RUN_TRAINING = True

EXPERIMENT_ID = "pipeline-smoke-test"

# Frequent training knobs live here.
TRAIN_OVERRIDES = {
    "epochs": 10,
    "batch": 16,
    "imgsz": 640,
    "workers": 4,
    # "device": 0,
}

# Optional eval-only overrides (useful after kernel restart).
EVAL_RUN_DIR = r"C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253"
EVAL_DATA_YAML = r"C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml"

# Dataset source is read from config/.env (workspace/project/version/format).
notes = (
    f"model={MODEL_NAME}; run_training={RUN_TRAINING}; "
    f"roboflow={config.ROBOFLOW_WORKSPACE}/{config.ROBOFLOW_PROJECT}:v{config.ROBOFLOW_VERSION}; "
    f"train_overrides={TRAIN_OVERRIDES}"
)

cfg = {
    "experiment_id": EXPERIMENT_ID,
    "notes": notes,
    "yolo_overrides": TRAIN_OVERRIDES,
}

print("model:", MODEL_NAME)
print("run training:", RUN_TRAINING)
print("roboflow dataset:", f"{config.ROBOFLOW_WORKSPACE}/{config.ROBOFLOW_PROJECT}:v{config.ROBOFLOW_VERSION}")
print("train overrides:", TRAIN_OVERRIDES)
print("eval override run_dir:", EVAL_RUN_DIR)
print("eval override data_yaml:", EVAL_DATA_YAML)

model: yolo
run training: True
dataset target: {'workspace': 'bus-project-frdgz', 'project': 'passenger-detection-on-a-bus-qgljh', 'version': 1, 'fmt': 'yolov11', 'force_download': True}
train overrides: {'epochs': 10, 'batch': 16, 'imgsz': 640, 'workers': 4}
eval override run_dir: C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253
eval override data_yaml: C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml


## 3) Download Dataset

In [8]:
import data

data_yaml = data.download_dataset(force_download=True)
print("dataset:", data_yaml)

data:     downloading bus-project-frdgz/passenger-detection-on-a-bus-qgljh v1 (yolov11) -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1
loading Roboflow workspace...
loading Roboflow project...
data:     dataset ready -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml
dataset: C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml


## 4) Train (Optional)

Set `RUN_TRAINING = False` in config to skip training and run eval-only.

In [9]:
import train
from datetime import datetime
from pathlib import Path

if RUN_TRAINING:
    run_dir = train.run(model_name=MODEL_NAME, cfg=cfg, data_yaml=data_yaml)
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"{EXPERIMENT_ID}_evalonly_{ts}"
    run_dir = (Path(config.RUNS_DIR) / run_name).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)
    print("train: skipped (eval-only mode)")

print("run_dir:", run_dir)

train:    run dir -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253


wandb:    run 'pipeline-smoke-test_20260513_190253' -> https://wandb.ai/cristianmaza/bus-passenger-count/runs/ttlvivau
yolo:     loading base weights -> yolo11m.pt
yolo:     training for 10 epochs on C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml
Ultralytics 8.4.50  Python-3.12.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end

## 5) Evaluate

Runs evaluation and uploads metrics + sample predictions to wandb.

In [9]:
from pathlib import Path
import eval as eval_mod

resolved_run_dir = Path(EVAL_RUN_DIR).resolve() if EVAL_RUN_DIR else globals().get("run_dir")
resolved_data_yaml = Path(EVAL_DATA_YAML).resolve() if EVAL_DATA_YAML else globals().get("data_yaml")

if resolved_run_dir is None:
    raise RuntimeError(
        "run_dir is missing. Run the Train cell first, or set EVAL_RUN_DIR in config."
    )
if resolved_data_yaml is None:
    raise RuntimeError(
        "data_yaml is missing. Run the Download Dataset cell first, or set EVAL_DATA_YAML in config."
    )

eval_weights = None if RUN_TRAINING else config.MODEL_WEIGHTS[MODEL_NAME]

metrics = eval_mod.run(
    run_dir=resolved_run_dir,
    data_yaml=resolved_data_yaml,
    model_name=MODEL_NAME,
    n_samples=10,
    cfg=cfg,
    weights=eval_weights,
)
metrics

eval:     weights -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253\yolo\weights\best.pt
yolo:     loading weights for eval -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253\yolo\weights\best.pt
Ultralytics 8.4.50  Python-3.12.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2418.71045.8 MB/s, size: 472.7 KB)
val: Scanning C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\test\labels.cache... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.4s3.9s
        

wandb:    logged 10 images under 'test/predictions'


final/test/fitness,▁
final/test/metrics/mAP50(B),▁
final/test/metrics/mAP50-95(B),▁
final/test/metrics/precision(B),▁
final/test/metrics/recall(B),▁
test/fitness,▁
test/metrics/mAP50(B),▁
test/metrics/mAP50-95(B),▁
test/metrics/precision(B),▁
test/metrics/recall(B),▁
final/test/fitness,0.4892


wandb:    run finished


{'test/metrics/precision(B)': 0.8748774492178799,
 'test/metrics/recall(B)': 0.8685446009389671,
 'test/metrics/mAP50(B)': 0.9359662093203561,
 'test/metrics/mAP50-95(B)': 0.4892042724812895,
 'test/fitness': 0.4892042724812895}